# L2d: Build and Test a Unicode Character Table

L2c introduced ASCII, Unicode code points, hexadecimal notation, and UTF-8 strings. In this lab, we use those ideas to build a function that converts a technical label into a table with one row for each character.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Distinguish character count from byte count:__ Compare the characters and UTF-8 bytes in a technical label and explain why a Unicode string may require more bytes than characters.
> * __Build a table by iterating over a string:__ Implement a documented function that records character positions, decimal code points, hexadecimal code points, and UTF-8 byte counts without assuming dense integer string indices.
> * __Test text-processing behavior:__ Use ASCII text, Unicode text, an empty string, and an invalid input to verify the complete function interface.

___

## Setup, Data, and Prerequisites

The setup file activates the course environment, loads the student implementation from [`src/Compute.jl`](src/Compute.jl), and imports `DataFrames` and `Test`.

[The `include(...)` function](https://docs.julialang.org/en/v1/base/base/#include) evaluates `Include.jl` in the notebook's global scope.

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # load the course environment and L2d source

___

## Task 1: Examine the characters and bytes in a technical label

Consider the label `ΔP = 25 kPa at 80 °C`. It contains common process notation, including the Greek capital delta and the degree symbol. Julia stores a `String` as UTF-8 code units, and each UTF-8 code unit is one byte. ASCII characters require one byte each, while `Δ` and `°` require more than one byte.

[The `length(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.length) counts characters when applied to this string. [The `ncodeunits(...)` function](https://docs.julialang.org/en/v1/base/strings/#Base.ncodeunits) counts its UTF-8 bytes. [The `collect(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.collect-Tuple%7BAny%7D) produces a `Vector{Char}`, while [`codeunits(...)`](https://docs.julialang.org/en/v1/base/strings/#Base.codeunits) exposes the stored byte values.

In [ ]:
technical_label = "ΔP = 25 kPa at 80 °C"

# Collect the logical characters separately from the underlying UTF-8 bytes.
label_characters = collect(technical_label)
label_bytes = collect(codeunits(technical_label))

# Compare the two counts and retain both representations for inspection.
(
    character_count = length(technical_label),
    byte_count = ncodeunits(technical_label),
    characters = label_characters,
    bytes = label_bytes,
)

The two counts differ because the string contains multibyte characters. This also explains why a Julia `String` should not be processed with `for index in 1:length(text)`. Integer positions inside a multibyte character are not valid string indices.

Iterating directly over the string returns complete `Char` values. Pairing that iteration with `enumerate(...)` provides the consecutive character positions needed for the output table.

In [ ]:
# Record the first five character positions without indexing into the String.
first_five_characters = collect(Iterators.take(enumerate(technical_label), 5))

___

## Task 2: Implement the character table function

Open [`src/Compute.jl`](src/Compute.jl) and complete its three `TODO` sections. The `character_table(text)` function returns a `DataFrame` with the following columns:

| Column | Meaning |
|:--|:--|
| `position` | Consecutive character position beginning at 1. |
| `character` | The Julia `Char` value at that position. |
| `decimal_codepoint` | The character's Unicode code point as a base-10 integer. |
| `unicode_codepoint` | The same code point in uppercase `U+XXXX` notation. |
| `utf8_byte_count` | Number of UTF-8 bytes used to store that character. |

Allocate typed columns before the loop so an empty input returns an empty table with the same schema as a populated table. Iterate with `enumerate(text)`, calculate the fields for one character, and add one complete row to the table. The function should reject a non-string input with an `ArgumentError`.

After completing the source file, restart the kernel and run the notebook from the beginning so Julia loads the revised module. The next cell raises a direct implementation error until all three `TODO` sections are complete.

In [ ]:
# Build the complete character table for the technical label.
technical_label_table = character_table(technical_label)
technical_label_table

The first row should describe `Δ` as decimal code point `916`, hexadecimal code point `U+0394`, and a two-byte UTF-8 character. The degree symbol should appear later as `U+00B0`, also with a two-byte UTF-8 representation. ASCII letters, digits, spaces, and punctuation should each use one byte.

In [ ]:
# Select the two non-ASCII rows for a direct comparison.
non_ascii_rows = filter(row -> row.utf8_byte_count > 1, technical_label_table)

___

## Task 3: Test the complete interface

The regression tests use Julia's [`Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/) to verify the table schema and the behavior for several input categories.

| Test tool | Purpose in this lab |
|:--|:--|
| [`@testset`](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@testset) | Groups the character-table checks and prints one summary. |
| [`@test`](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@test) | Verifies table columns, code points, byte counts, and empty-string behavior. |
| [`@test_throws`](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@test_throws) | Verifies that a non-string input raises the documented exception type. |

The ASCII case verifies the simplest storage pattern. The Unicode case verifies multibyte characters and code-point formatting. The empty-string case verifies that the function returns the documented schema even when there are no rows.

In [ ]:
@testset "Unicode character table interface" begin
    # Verify the schema and contents for text containing only ASCII characters.
    ascii_table = character_table("P101")
    @test names(ascii_table) == [
        "position",
        "character",
        "decimal_codepoint",
        "unicode_codepoint",
        "utf8_byte_count",
    ]
    @test ascii_table.position == [1, 2, 3, 4]
    @test ascii_table.character == ['P', '1', '0', '1']
    @test ascii_table.utf8_byte_count == [1, 1, 1, 1]

    # Verify code points and byte counts for Greek and technical symbols.
    unicode_table = character_table("ΔP °C")
    @test unicode_table.character == ['Δ', 'P', ' ', '°', 'C']
    @test unicode_table.decimal_codepoint[[1, 4]] == [916, 176]
    @test unicode_table.unicode_codepoint[[1, 4]] == ["U+0394", "U+00B0"]
    @test unicode_table.utf8_byte_count == [2, 1, 1, 2, 1]

    # Verify the row count and typed schema for an empty string.
    empty_table = character_table("")
    @test nrow(empty_table) == 0
    @test eltype(empty_table.character) == Char
    @test eltype(empty_table.unicode_codepoint) == String

    # Verify that an input outside the documented interface is rejected.
    @test_throws ArgumentError character_table(42)
end

___

## Summary

In this lab, we examined the UTF-8 representation of a technical label, implemented a function that converts text into a Unicode character table, and tested the complete interface.

> __Key Takeaways:__
>
> * **Characters and bytes are different units:** A Unicode string can contain fewer characters than bytes because UTF-8 uses multiple bytes for many non-ASCII code points.
> * **Direct iteration preserves character boundaries:** Iterating over a string returns complete `Char` values, while assuming that every integer is a valid string index can place an index inside a multibyte character.
> * **An empty result still needs a defined schema:** Allocating typed table columns before processing the input gives empty and populated results the same documented structure.

These practices apply when technical labels, units, symbols, or data files contain text that extends beyond basic ASCII.

___